<a href="https://colab.research.google.com/github/Juanma-dev-tech/Procesamiento__del_Habla/blob/main/TP5/Y%C3%B1iguez_TP5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP5 – Evolución de un chatbot de recuperación mediante embeddings, base vectorial y LLM

Este trabajo parte de la implementación realizada en el TP4, donde se desarrollaron dos chatbots basados en recuperación de información para atención de estudiantes universitarios.

En el TP4 se compararon dos enfoques:

- TF-IDF + similitud del coseno.
- Embeddings preentrenados mediante spaCy.

Como evolución del trabajo anterior, en este TP se incorporará:

- Una base de datos vectorial para almacenar y recuperar embeddings.
- Dos modelos de embeddings.
- Un modelo LLM de HuggingFace.
- Evaluación comparativa de los embeddings utilizados.

# Antecedentes TP4

Las siguientes secciones corresponden a la implementación realizada en el TP4 y se mantienen como referencia para contextualizar la evolución hacia una arquitectura basada en bases de datos vectoriales y modelos LLM.

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [ ]:
!pip install spacy --quiet
!python -m spacy download es_core_news_sm --quiet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 67.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Actividades



### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado eligió.

### 4) Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.



### No olvides:

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes





# Aplicación elegida

Para este trabajo se desarrollará un chatbot basado en recuperación de información orientado a la atención de estudiantes universitarios.

El objetivo del chatbot será responder preguntas frecuentes relacionadas con inscripciones, exámenes, campus virtual, certificados, becas y trámites académicos.

Se implementarán dos enfoques distintos:

- Un chatbot utilizando TF-IDF y similitud del coseno.
- Un chatbot utilizando embeddings preentrenados en español.

Finalmente se comparará el rendimiento de ambos modelos para analizar sus diferencias y limitaciones.

## Creación del dataset

Se elaboró un dataset propio compuesto por preguntas y respuestas frecuentes relacionadas con la atención a estudiantes universitarios.

El dataset contiene consultas habituales sobre:
- inscripciones,
- exámenes,
- becas,
- campus virtual,
- certificados,
- correlatividades,
- y trámites académicos.

Este conjunto de datos será utilizado para implementar ambos chatbots basados en recuperación de información.

In [ ]:
import pandas as pd

In [ ]:
datos = {
    "pregunta": [
        "¿Cómo me inscribo a materias?",
        "¿Cuándo comienzan las clases?",
        "¿Cómo recupero mi contraseña del campus virtual?",
        "¿Dónde puedo ver mis notas?",
        "¿Cómo solicito un certificado de alumno regular?",
        "¿Cuándo son los exámenes finales?",
        "¿Cómo solicito una beca?",
        "¿Qué documentación necesito para inscribirme?",
        "¿Dónde veo mi horario de cursada?",
        "¿Cómo puedo dar de baja una materia?",
        "¿Cómo accedo al campus virtual?",
        "¿Qué pasa si desapruebo un final?",
        "¿Cuántas materias puedo cursar?",
        "¿Cómo solicito equivalencias?",
        "¿Dónde consulto el calendario académico?",
        "¿Cómo contacto a un profesor?",
        "¿Qué es una correlativa?",
        "¿Cómo justifico una inasistencia?",
        "¿Dónde se publican las fechas de inscripción?",
        "¿Cómo actualizo mis datos personales?"
    ],

    "respuesta": [
        "Puedes inscribirte a materias desde el sistema de autogestión estudiantil.",
        "Las clases comienzan según el calendario académico publicado por la universidad.",
        "Puedes recuperar tu contraseña desde la opción 'Olvidé mi contraseña' del campus virtual.",
        "Las notas pueden consultarse desde el sistema de alumnos.",
        "El certificado puede solicitarse desde autogestión en la sección de certificados.",
        "Las fechas de finales se publican en el calendario académico.",
        "Las becas pueden solicitarse desde el portal de bienestar estudiantil.",
        "Necesitas DNI y documentación académica previa para inscribirte.",
        "El horario de cursada se encuentra disponible en el sistema de alumnos.",
        "La baja de materias puede realizarse desde autogestión.",
        "Puedes acceder al campus virtual con tu usuario institucional.",
        "Podrás volver a inscribirte al examen final en otra fecha.",
        "La cantidad depende del plan de estudios y correlatividades.",
        "Las equivalencias deben solicitarse en la secretaría académica.",
        "El calendario académico está disponible en la página institucional.",
        "Puedes contactar profesores mediante el correo institucional.",
        "Una correlativa es una materia que debe aprobarse previamente.",
        "Las inasistencias se justifican presentando documentación válida.",
        "Las fechas se publican en la web de alumnos.",
        "Los datos personales pueden actualizarse desde autogestión."
    ]
}

In [ ]:
df = pd.DataFrame(datos)

df.head()

,pregunta,respuesta
0,¿Cómo me inscribo a materias?,Puedes inscribirte a materias desde el sistema...
1,¿Cuándo comienzan las clases?,Las clases comienzan según el calendario acadé...
2,¿Cómo recupero mi contraseña del campus virtual?,Puedes recuperar tu contraseña desde la opción...
3,¿Dónde puedo ver mis notas?,Las notas pueden consultarse desde el sistema ...
4,¿Cómo solicito un certificado de alumno regular?,El certificado puede solicitarse desde autoges...


# Chatbot utilizando TF-IDF y similitud del coseno

En esta sección se implementará un chatbot basado en recuperación de información utilizando TF-IDF (Term Frequency - Inverse Document Frequency) y similitud del coseno.

TF-IDF permite transformar preguntas en vectores numéricos ponderando la importancia de las palabras dentro del conjunto de textos.

Luego, mediante similitud del coseno, se comparará la consulta del usuario contra las preguntas almacenadas en el dataset para recuperar la respuesta más similar.

In [ ]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df["pregunta"])

In [ ]:
print("Dimensiones de la matriz TF-IDF:")
print(tfidf_matrix.shape)

Dimensiones de la matriz TF-IDF:
(20, 68)


In [ ]:
def chatbot_tfidf(consulta_usuario):

    consulta_vec = vectorizer.transform([consulta_usuario])

    similitudes = cosine_similarity(
        consulta_vec,
        tfidf_matrix
    ).flatten()

    indice = similitudes.argmax()

    respuesta = df.iloc[indice]["respuesta"]

    return respuesta

## Preguntas utilizadas para probar el funcionamiento

Las siguientes consultas fueron utilizadas para evaluar ambos chatbots. Algunas fueron reformuladas respecto del dataset original para analizar cómo responde cada enfoque ante preguntas similares pero expresadas con otras palabras.

In [ ]:
preguntas_prueba = [
    "quiero anotarme a una materia",
    "como veo mis calificaciones",
    "olvide la contraseña del campus",
    "donde consulto fechas de finales",
    "como pedir una beca"
]

In [ ]:
for pregunta in preguntas_prueba:

    print("Pregunta:", pregunta)

    print("Respuesta:", chatbot_tfidf(pregunta))

    print("-" * 60)

Pregunta: quiero anotarme a una materia
Respuesta: La baja de materias puede realizarse desde autogestión.
------------------------------------------------------------
Pregunta: como veo mis calificaciones
Respuesta: El horario de cursada se encuentra disponible en el sistema de alumnos.
------------------------------------------------------------
Pregunta: olvide la contraseña del campus
Respuesta: Puedes recuperar tu contraseña desde la opción 'Olvidé mi contraseña' del campus virtual.
------------------------------------------------------------
Pregunta: donde consulto fechas de finales
Respuesta: Las fechas se publican en la web de alumnos.
------------------------------------------------------------
Pregunta: como pedir una beca
Respuesta: Las becas pueden solicitarse desde el portal de bienestar estudiantil.
------------------------------------------------------------


## Análisis inicial del chatbot TF-IDF

El chatbot basado en TF-IDF logró responder correctamente varias consultas relacionadas con estudiantes universitarios.

Sin embargo, este enfoque depende principalmente de las palabras utilizadas en la consulta. Cuando el usuario formula preguntas utilizando palabras muy diferentes a las almacenadas en el dataset, el modelo puede presentar dificultades para recuperar la respuesta correcta.

Esto ocurre porque TF-IDF trabaja sobre frecuencia e importancia de términos, pero no comprende completamente el significado semántico de las oraciones.

In [ ]:
resultados_tfidf = pd.DataFrame({
    "Pregunta": preguntas_prueba,
    "Resultado": [
        "Incorrecto",
        "Incorrecto",
        "Correcto",
        "Parcialmente correcto",
        "Correcto"
    ]
})

resultados_tfidf

,Pregunta,Resultado
0,quiero anotarme a una materia,Incorrecto
1,como veo mis calificaciones,Incorrecto
2,olvide la contraseña del campus,Correcto
3,donde consulto fechas de finales,Parcialmente correcto
4,como pedir una beca,Correcto


## Observaciones sobre TF-IDF

El modelo basado en TF-IDF respondió correctamente consultas donde las palabras utilizadas por el usuario coincidían directamente con las preguntas almacenadas.

Sin embargo, presentó dificultades cuando el usuario reformuló preguntas utilizando sinónimos o expresiones diferentes.

Por ejemplo:
- "anotarme" no fue asociado correctamente con "inscribirme".
- "calificaciones" no fue relacionado correctamente con "notas".

Esto evidencia que TF-IDF depende principalmente de coincidencias léxicas y no comprende completamente el significado semántico de las consultas.

# Chatbot utilizando embeddings

En esta sección se implementará un chatbot basado en embeddings preentrenados utilizando spaCy.

A diferencia de TF-IDF, los embeddings representan palabras y oraciones mediante vectores densos que capturan relaciones semánticas entre términos.

Esto permite que el modelo pueda reconocer similitudes de significado incluso cuando las palabras utilizadas no son exactamente iguales.

In [ ]:
!python -m spacy download es_core_news_md --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 18.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Generación de embeddings

Para este chatbot se utilizará el modelo preentrenado `es_core_news_md` de spaCy.

Este modelo permite representar preguntas mediante vectores semánticos densos, facilitando la comparación entre consultas incluso cuando utilizan palabras diferentes pero con significados similares.

In [ ]:
import spacy

nlp = spacy.load("es_core_news_md")

In [ ]:
embeddings_preguntas = []

for pregunta in df["pregunta"]:

    doc = nlp(pregunta)

    embeddings_preguntas.append(doc.vector)

In [ ]:
import numpy as np

## Implementación del chatbot basado en embeddings

La siguiente función transforma la consulta del usuario en un embedding y calcula la similitud del coseno contra las preguntas almacenadas en el dataset.

La respuesta asociada a la pregunta más similar será devuelta por el chatbot.

In [ ]:
def chatbot_embeddings(consulta_usuario):

    consulta_vec = nlp(consulta_usuario).vector

    similitudes = []

    for emb in embeddings_preguntas:

        similitud = cosine_similarity(
            [consulta_vec],
            [emb]
        )[0][0]

        similitudes.append(similitud)

    indice = np.argmax(similitudes)

    respuesta = df.iloc[indice]["respuesta"]

    return respuesta

In [ ]:
for pregunta in preguntas_prueba:

    print("Pregunta:", pregunta)

    print("Respuesta:", chatbot_embeddings(pregunta))

    print("-" * 60)

Pregunta: quiero anotarme a una materia
Respuesta: La baja de materias puede realizarse desde autogestión.
------------------------------------------------------------
Pregunta: como veo mis calificaciones
Respuesta: Las notas pueden consultarse desde el sistema de alumnos.
------------------------------------------------------------
Pregunta: olvide la contraseña del campus
Respuesta: La baja de materias puede realizarse desde autogestión.
------------------------------------------------------------
Pregunta: donde consulto fechas de finales
Respuesta: Las fechas se publican en la web de alumnos.
------------------------------------------------------------
Pregunta: como pedir una beca
Respuesta: Las becas pueden solicitarse desde el portal de bienestar estudiantil.
------------------------------------------------------------


## Evaluación del chatbot con embeddings

Se realizaron pruebas utilizando consultas reformuladas para evaluar la capacidad semántica del modelo.

Los resultados obtenidos muestran que el enfoque basado en embeddings logró mejorar algunas respuestas respecto a TF-IDF, especialmente en consultas donde las palabras utilizadas por el usuario diferían de las preguntas originales pero mantenían un significado similar.

Sin embargo, también se observaron algunos errores y confusiones en determinadas consultas, lo que evidencia que el rendimiento del modelo depende tanto de la calidad del embedding utilizado como del tamaño y variedad del dataset.

# Comparación entre ambos enfoques

A continuación se presenta una comparación entre los resultados obtenidos utilizando TF-IDF y embeddings en distintas consultas de prueba.

In [ ]:
comparacion = pd.DataFrame({

    "Pregunta": preguntas_prueba,

    "Resultado TF-IDF": [
        "Incorrecto",
        "Incorrecto",
        "Correcto",
        "Parcialmente correcto",
        "Correcto"
    ],

    "Resultado Embeddings": [
        "Incorrecto",
        "Correcto",
        "Incorrecto",
        "Parcialmente correcto",
        "Correcto"
    ]
})

comparacion

,Pregunta,Resultado TF-IDF,Resultado Embeddings
0,quiero anotarme a una materia,Incorrecto,Incorrecto
1,como veo mis calificaciones,Incorrecto,Correcto
2,olvide la contraseña del campus,Correcto,Incorrecto
3,donde consulto fechas de finales,Parcialmente correcto,Parcialmente correcto
4,como pedir una beca,Correcto,Correcto


## Análisis comparativo

El chatbot basado en TF-IDF mostró mejores resultados cuando las consultas del usuario contenían palabras similares a las preguntas almacenadas en el dataset.

Por otro lado, el chatbot basado en embeddings logró interpretar mejor algunas relaciones semánticas entre términos diferentes pero conceptualmente cercanos, como "calificaciones" y "notas".

Sin embargo, ambos enfoques presentaron limitaciones en determinadas consultas. Esto demuestra que el rendimiento depende tanto del método de vectorización como de la calidad y variedad del dataset utilizado.

En datasets pequeños, los embeddings pueden generar asociaciones semánticas incorrectas o ambiguas, mientras que TF-IDF puede fallar cuando el usuario utiliza expresiones diferentes a las almacenadas originalmente.

# Conclusiones

En este trabajo se desarrollaron dos chatbots basados en recuperación de información utilizando diferentes técnicas de vectorización de texto.

El primer enfoque utilizó TF-IDF y similitud del coseno para recuperar respuestas a partir de coincidencias entre términos importantes. Este método resultó simple y efectivo cuando las consultas compartían palabras similares con las preguntas originales.

El segundo enfoque utilizó embeddings preentrenados en español mediante spaCy. Este modelo permitió representar mejor ciertas relaciones semánticas entre palabras y mejorar algunas respuestas donde las consultas estaban reformuladas utilizando términos diferentes.

A pesar de ello, ambos modelos presentaron limitaciones y errores en determinadas situaciones, especialmente debido al tamaño reducido del dataset utilizado. Esto permitió observar cómo distintos enfoques de representación textual pueden influir directamente en el comportamiento de un chatbot basado en recuperación.

Durante la realización de esta actividad se pudo comprender mejor el funcionamiento de TF-IDF, embeddings y similitud del coseno aplicados al procesamiento de lenguaje natural.

Además, se pudo observar cómo el tamaño y diversidad del dataset influyen directamente en el rendimiento de sistemas basados en recuperación de información.

# Fuentes utilizadas

- Material de la materia Procesamiento del Habla.
- Notebook TP3-chatbot-retrieval-template-parte-1.ipynb.
- Documentación oficial de spaCy:
https://spacy.io/

- Documentación oficial de scikit-learn:
https://scikit-learn.org/


# Herramientas de apoyo utilizadas

Durante la realización del trabajo se utilizaron herramientas de inteligencia artificial como apoyo complementario para consultas conceptuales y revisión de explicaciones relacionadas con procesamiento de lenguaje natural y embeddings.

# TP5 - Implementación


# 1) CONSIGNAS

## a) Creación del conjunto de datos de evaluación

Además del dataset original que creó, debe crear un dataset de prueba o evaluación con la misma lógica: preguntas y respuestas.

### Desarrollo

Además del dataset original utilizado como base de conocimiento, se construyó un dataset de evaluación independiente.

Las preguntas fueron reformuladas respecto de las utilizadas en el TP4, manteniendo el mismo significado pero utilizando expresiones diferentes, sinónimos o construcciones alternativas.

El objetivo es evaluar la capacidad de recuperación semántica del chatbot y comparar el desempeño de los distintos modelos de embeddings seleccionados.

In [ ]:
datos_evaluacion = {
    "pregunta": [
        "¿Dónde me anoto para cursar una materia?",
        "¿En qué fecha arrancan las cursadas?",
        "Olvidé mi clave del campus, ¿cómo la recupero?",
        "¿Cómo consulto mis calificaciones?",
        "¿Dónde solicito una constancia de alumno regular?",
        "¿Cuándo se toman los finales?",
        "¿Dónde puedo pedir una beca para estudiar?",
        "¿Qué papeles tengo que presentar para inscribirme?",
        "¿Cómo veo los horarios de mis materias?",
        "¿Cómo hago para abandonar una materia?",
        "¿Cuál es el acceso a las aulas virtuales?",
        "¿Qué sucede si no apruebo un examen final?",
        "¿Existe un límite de materias para cursar?",
        "¿Cómo tramito equivalencias de materias?",
        "¿Dónde encuentro el cronograma académico?",
        "¿Cómo puedo comunicarme con un docente?",
        "¿Qué significa que una materia sea correlativa?",
        "¿Cómo justifico una falta a clase?",
        "¿Dónde aparecen las fechas para anotarse a materias?",
        "¿Cómo modifico mi información personal?"
    ],

    "respuesta_esperada": [
        "Puedes inscribirte a materias desde el sistema de autogestión estudiantil.",
        "Las clases comienzan según el calendario académico publicado por la universidad.",
        "Puedes recuperar tu contraseña desde la opción 'Olvidé mi contraseña' del campus virtual.",
        "Las notas pueden consultarse desde el sistema de alumnos.",
        "El certificado puede solicitarse desde autogestión en la sección de certificados.",
        "Las fechas de finales se publican en el calendario académico.",
        "Las becas pueden solicitarse desde el portal de bienestar estudiantil.",
        "Necesitas DNI y documentación académica previa para inscribirte.",
        "El horario de cursada se encuentra disponible en el sistema de alumnos.",
        "La baja de materias puede realizarse desde autogestión.",
        "Puedes acceder al campus virtual con tu usuario institucional.",
        "Podrás volver a inscribirte al examen final en otra fecha.",
        "La cantidad depende del plan de estudios y correlatividades.",
        "Las equivalencias deben solicitarse en la secretaría académica.",
        "El calendario académico está disponible en la página institucional.",
        "Puedes contactar profesores mediante el correo institucional.",
        "Una correlativa es una materia que debe aprobarse previamente.",
        "Las inasistencias se justifican presentando documentación válida.",
        "Las fechas se publican en la web de alumnos.",
        "Los datos personales pueden actualizarse desde autogestión."
    ]
}

df_evaluacion = pd.DataFrame(datos_evaluacion)

print("Cantidad de preguntas de evaluación:", len(df_evaluacion))
df_evaluacion.head()

Cantidad de preguntas de evaluación: 20


,pregunta,respuesta_esperada
0,¿Dónde me anoto para cursar una materia?,Puedes inscribirte a materias desde el sistema...
1,¿En qué fecha arrancan las cursadas?,Las clases comienzan según el calendario acadé...
2,"Olvidé mi clave del campus, ¿cómo la recupero?",Puedes recuperar tu contraseña desde la opción...
3,¿Cómo consulto mis calificaciones?,Las notas pueden consultarse desde el sistema ...
4,¿Dónde solicito una constancia de alumno regular?,El certificado puede solicitarse desde autoges...


## b) Debe elegir un modelo LLM de HuggingFace y al menos dos modelos de embeddings. Justifique su elección.

### Desarrollo

Para la implementación del chatbot se seleccionará un modelo LLM de Hugging Face y dos modelos de embeddings, los cuales serán comparados para determinar cuál ofrece mejores resultados en la recuperación de información.

### Modelos de embeddings seleccionados

Para comparar el desempeño del chatbot se seleccionaron dos modelos de embeddings:

**1. all-MiniLM-L6-v2**

Este modelo se utiliza como referencia porque es el modelo por defecto de ChromaDB cuando no se especifica una función de embedding personalizada. Se eligió por su integración directa con la base vectorial y por su bajo costo computacional, lo cual resulta conveniente para trabajar en Google Colab.

**2. paraphrase-multilingual-MiniLM-L12-v2**

Este modelo fue seleccionado porque está orientado a tareas de similitud semántica y soporta múltiples idiomas, incluido el español. Se espera que pueda interpretar mejor preguntas reformuladas con sinónimos o expresiones diferentes a las del dataset original.

La comparación entre ambos modelos permitirá analizar cuál recupera con mayor precisión las respuestas esperadas para el dominio de atención a estudiantes universitarios.

### Modelo LLM seleccionado

Para la generación de respuestas se seleccionó el modelo **microsoft/Phi-3-mini-4k-instruct**, disponible en Hugging Face.

Se eligió este modelo por ser un LLM moderno optimizado para tareas de conversación, preguntas y respuestas e instrucciones. Además, ofrece un buen equilibrio entre calidad de respuesta y requerimientos computacionales, permitiendo su utilización en entornos como Google Colab.

Este modelo será utilizado junto con la base vectorial y los embeddings seleccionados para generar las respuestas del chatbot.

## c) Implemente una clase ChatBot usando lo elegido en b).

### Desarrollo

La clase `ChatBot` implementará una arquitectura basada en recuperación aumentada por generación.

Primero, el dataset original del TP4 será cargado como base de conocimiento en ChromaDB. Luego, ante una pregunta del usuario, el sistema recuperará el contexto más relevante utilizando embeddings. Finalmente, el modelo LLM seleccionado generará una respuesta tomando como base ese contexto recuperado.

De esta forma, el chatbot no responde únicamente copiando una respuesta fija del dataset, sino que genera una respuesta utilizando la información recuperada desde la base vectorial.

In [ ]:
!pip install chromadb sentence-transformers transformers accelerate --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

print("Librerías importadas correctamente")

Librerías importadas correctamente


In [ ]:
# Modelos seleccionados para TP5

embedding_1 = "all-MiniLM-L6-v2"
embedding_2 = "paraphrase-multilingual-MiniLM-L12-v2"

modelo_llm = "microsoft/Phi-3-mini-4k-instruct"

print("Embedding 1:", embedding_1)
print("Embedding 2:", embedding_2)
print("LLM:", modelo_llm)

Embedding 1: all-MiniLM-L6-v2
Embedding 2: paraphrase-multilingual-MiniLM-L12-v2
LLM: microsoft/Phi-3-mini-4k-instruct


In [ ]:
client = chromadb.EphemeralClient()

collection = client.create_collection(
    name="chatbot_tp5"
)

print("Colección creada correctamente")

Colección creada correctamente


In [ ]:
documentos = [
    f"Pregunta: {row['pregunta']}\nRespuesta: {row['respuesta']}"
    for _, row in df.iterrows()
]

metadatos = [
    {
        "pregunta": row["pregunta"],
        "respuesta": row["respuesta"]
    }
    for _, row in df.iterrows()
]

ids = [f"doc_{i}" for i in range(len(df))]

collection.add(
    documents=documentos,
    metadatas=metadatos,
    ids=ids
)

print("Cantidad de documentos cargados en ChromaDB:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 32.4MiB/s]


Cantidad de documentos cargados en ChromaDB: 20


In [ ]:
resultado = collection.query(
    query_texts=["¿Dónde me anoto para cursar una materia?"],
    n_results=3
)

print(resultado["documents"][0])

['Pregunta: ¿Cuántas materias puedo cursar?\nRespuesta: La cantidad depende del plan de estudios y correlatividades.', 'Pregunta: ¿Cómo me inscribo a materias?\nRespuesta: Puedes inscribirte a materias desde el sistema de autogestión estudiantil.', 'Pregunta: ¿Cómo puedo dar de baja una materia?\nRespuesta: La baja de materias puede realizarse desde autogestión.']


### Comparación de modelos de embeddings

Con el objetivo de comparar el desempeño de distintos modelos de embeddings, se implementará una segunda colección de ChromaDB utilizando el modelo paraphrase-multilingual-MiniLM-L12-v2. De esta manera será posible realizar consultas equivalentes sobre ambas colecciones y analizar las diferencias en los resultados recuperados.

In [ ]:
# Crear función de embedding personalizada para el segundo modelo

embedding_func_2 = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=embedding_2
)

# Crear segunda colección usando paraphrase-multilingual-MiniLM-L12-v2

collection_2 = client.create_collection(
    name="chatbot_tp5_embedding_2",
    embedding_function=embedding_func_2
)

collection_2.add(
    documents=documentos,
    metadatas=metadatos,
    ids=ids
)

print("Cantidad de documentos cargados en ChromaDB con embedding 2:", collection_2.count())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cantidad de documentos cargados en ChromaDB con embedding 2: 20


### Implementación de la clase ChatBot

A continuación se implementa una clase ChatBot que integra una base vectorial ChromaDB y un modelo LLM de Hugging Face. El chatbot recuperará el contexto más relevante utilizando embeddings y posteriormente generará una respuesta a partir de dicho contexto.

In [ ]:
class ChatBot:
    def __init__(self, collection, modelo_llm):
        self.collection = collection
        self.modelo_llm = modelo_llm

    def recuperar_contexto(self, pregunta, n_resultados=3):
        resultado = self.collection.query(
            query_texts=[pregunta],
            n_results=n_resultados
        )

        contextos = resultado["documents"][0]
        metadatas = resultado["metadatas"][0]

        contexto = "\n\n".join(contextos)

        return contexto, metadatas

    def generar_prompt(self, pregunta, contexto):
        prompt = f"""
Eres un asistente para estudiantes universitarios.

Usa solamente la información del CONTEXTO.
No inventes pasos ni información adicional.
Si la respuesta no está en el contexto, responde: "No tengo información suficiente para responder."

CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}

RESPUESTA:
"""
        return prompt

    def responder(self, pregunta):
        contexto, metadatas = self.recuperar_contexto(pregunta)

        prompt = self.generar_prompt(pregunta, contexto)

        salida = generador_texto(
            prompt,
            max_new_tokens=50,
            do_sample=False,
            return_full_text=False
        )[0]["generated_text"]

        respuesta_limpia = salida.strip().split("\n")[0]

        return {
            "pregunta_usuario": pregunta,
            "contexto_recuperado": contexto,
            "preguntas_recuperadas": [m["pregunta"] for m in metadatas],
            "respuestas_recuperadas": [m["respuesta"] for m in metadatas],
            "respuesta_generada": respuesta_limpia
        }

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(modelo_llm)

model = AutoModelForCausalLM.from_pretrained(
    modelo_llm,
    torch_dtype=torch.float16,
    device_map="auto"
)

generador_texto = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Modelo LLM cargado correctamente")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Modelo LLM cargado correctamente


In [ ]:
chatbot = ChatBot(
    collection=collection,
    modelo_llm=modelo_llm
)

print("ChatBot inicializado correctamente")

ChatBot inicializado correctamente


In [ ]:
respuesta = chatbot.responder(
    "¿Dónde me anoto para cursar una materia?"
)

respuesta

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'pregunta_usuario': '¿Dónde me anoto para cursar una materia?',
 'contexto_recuperado': 'Pregunta: ¿Cuántas materias puedo cursar?\nRespuesta: La cantidad depende del plan de estudios y correlatividades.\n\nPregunta: ¿Cómo me inscribo a materias?\nRespuesta: Puedes inscribirte a materias desde el sistema de autogestión estudiantil.\n\nPregunta: ¿Cómo puedo dar de baja una materia?\nRespuesta: La baja de materias puede realizarse desde autogestión.',
 'preguntas_recuperadas': ['¿Cuántas materias puedo cursar?',
  '¿Cómo me inscribo a materias?',
  '¿Cómo puedo dar de baja una materia?'],
 'respuestas_recuperadas': ['La cantidad depende del plan de estudios y correlatividades.',
  'Puedes inscribirte a materias desde el sistema de autogestión estudiantil.',
  'La baja de materias puede realizarse desde autogestión.'],
 'respuesta_generada': 'Puedes anotarte para cursar una materia desde el sistema de autogestión estudiantil.'}

### Prueba de funcionamiento del chatbot

Se realizó una prueba inicial del chatbot utilizando la base de conocimiento creada a partir del dataset del TP4. El sistema recuperó información relevante mediante ChromaDB y posteriormente generó una respuesta utilizando el modelo LLM seleccionado.

## d) Pruebe el chatbot creado en c) con las preguntas de su dataset a)

Usando los modelos elegidos en b), observe las respuestas generadas por el chatbot comparando al menos dos modelos de embeddings.

Justifique y determine cuál elegiría para su aplicación.

### Desarrollo
Para evaluar el desempeño de los modelos de embeddings seleccionados, se realizarán consultas utilizando las preguntas del dataset de evaluación construido en el punto a).

Cada consulta será ejecutada sobre dos colecciones ChromaDB:

- Collection 1: all-MiniLM-L6-v2 (modelo por defecto de ChromaDB).
- Collection 2: paraphrase-multilingual-MiniLM-L12-v2.

Posteriormente se analizarán los documentos recuperados y las respuestas generadas para determinar cuál de los dos modelos resulta más adecuado para el dominio de atención a estudiantes universitarios.

In [ ]:
preguntas_prueba = [
    "¿Dónde me anoto para cursar una materia?",
    "¿Cómo consulto mis calificaciones?",
    "¿Dónde solicito una constancia de alumno regular?"
]

for pregunta in preguntas_prueba:

    resultado_1 = collection.query(
        query_texts=[pregunta],
        n_results=1
    )

    resultado_2 = collection_2.query(
        query_texts=[pregunta],
        n_results=1
    )

    print("=" * 80)
    print("PREGUNTA:", pregunta)

    print("\nEmbedding 1 (all-MiniLM-L6-v2):")
    print(resultado_1["documents"][0][0])

    print("\nEmbedding 2 (paraphrase-multilingual-MiniLM-L12-v2):")
    print(resultado_2["documents"][0][0])

    print()

PREGUNTA: ¿Dónde me anoto para cursar una materia?

Embedding 1 (all-MiniLM-L6-v2):
Pregunta: ¿Cuántas materias puedo cursar?
Respuesta: La cantidad depende del plan de estudios y correlatividades.

Embedding 2 (paraphrase-multilingual-MiniLM-L12-v2):
Pregunta: ¿Cómo me inscribo a materias?
Respuesta: Puedes inscribirte a materias desde el sistema de autogestión estudiantil.

PREGUNTA: ¿Cómo consulto mis calificaciones?

Embedding 1 (all-MiniLM-L6-v2):
Pregunta: ¿Cómo actualizo mis datos personales?
Respuesta: Los datos personales pueden actualizarse desde autogestión.

Embedding 2 (paraphrase-multilingual-MiniLM-L12-v2):
Pregunta: ¿Dónde puedo ver mis notas?
Respuesta: Las notas pueden consultarse desde el sistema de alumnos.

PREGUNTA: ¿Dónde solicito una constancia de alumno regular?

Embedding 1 (all-MiniLM-L6-v2):
Pregunta: ¿Cómo solicito un certificado de alumno regular?
Respuesta: El certificado puede solicitarse desde autogestión en la sección de certificados.

Embedding 2 (par

### Análisis de las pruebas iniciales (3 preguntas)

En las pruebas realizadas se observó que el modelo paraphrase-multilingual-MiniLM-L12-v2 recuperó correctamente las preguntas equivalentes del dataset original aun cuando las consultas fueron reformuladas utilizando expresiones diferentes.

Por el contrario, el modelo all-MiniLM-L6-v2 presentó dificultades para identificar algunas equivalencias semánticas, recuperando preguntas relacionadas pero no necesariamente las más adecuadas.

Estos resultados sugieren que paraphrase-multilingual-MiniLM-L12-v2 resulta más apropiado para aplicaciones donde los usuarios pueden formular la misma consulta de múltiples maneras.

In [ ]:
resultados = []

for pregunta, respuesta_esperada in zip(
    datos_evaluacion["pregunta"],
    datos_evaluacion["respuesta_esperada"]
):

    r1 = collection.query(
        query_texts=[pregunta],
        n_results=1
    )

    r2 = collection_2.query(
        query_texts=[pregunta],
        n_results=1
    )

    resultados.append({
        "pregunta": pregunta,
        "respuesta_esperada": respuesta_esperada,
        "embedding_1": r1["metadatas"][0][0]["pregunta"],
        "embedding_2": r2["metadatas"][0][0]["pregunta"]
    })

df_resultados = pd.DataFrame(resultados)

df_resultados.head()

,pregunta,respuesta_esperada,embedding_1,embedding_2
0,¿Dónde me anoto para cursar una materia?,Puedes inscribirte a materias desde el sistema...,¿Cuántas materias puedo cursar?,¿Cómo me inscribo a materias?
1,¿En qué fecha arrancan las cursadas?,Las clases comienzan según el calendario acadé...,¿Cuántas materias puedo cursar?,¿Cuándo comienzan las clases?
2,"Olvidé mi clave del campus, ¿cómo la recupero?",Puedes recuperar tu contraseña desde la opción...,¿Cómo recupero mi contraseña del campus virtual?,¿Cómo recupero mi contraseña del campus virtual?
3,¿Cómo consulto mis calificaciones?,Las notas pueden consultarse desde el sistema ...,¿Cómo actualizo mis datos personales?,¿Dónde puedo ver mis notas?
4,¿Dónde solicito una constancia de alumno regular?,El certificado puede solicitarse desde autoges...,¿Cómo solicito un certificado de alumno regular?,¿Cómo solicito un certificado de alumno regular?


In [ ]:
len(df_resultados)

20

In [ ]:
df_resultados

,pregunta,respuesta_esperada,embedding_1,embedding_2
0,¿Dónde me anoto para cursar una materia?,Puedes inscribirte a materias desde el sistema...,¿Cuántas materias puedo cursar?,¿Cómo me inscribo a materias?
1,¿En qué fecha arrancan las cursadas?,Las clases comienzan según el calendario acadé...,¿Cuántas materias puedo cursar?,¿Cuándo comienzan las clases?
2,"Olvidé mi clave del campus, ¿cómo la recupero?",Puedes recuperar tu contraseña desde la opción...,¿Cómo recupero mi contraseña del campus virtual?,¿Cómo recupero mi contraseña del campus virtual?
3,¿Cómo consulto mis calificaciones?,Las notas pueden consultarse desde el sistema ...,¿Cómo actualizo mis datos personales?,¿Dónde puedo ver mis notas?
4,¿Dónde solicito una constancia de alumno regular?,El certificado puede solicitarse desde autoges...,¿Cómo solicito un certificado de alumno regular?,¿Cómo solicito un certificado de alumno regular?
5,¿Cuándo se toman los finales?,Las fechas de finales se publican en el calend...,¿Cuándo son los exámenes finales?,¿Cuándo son los exámenes finales?
6,¿Dónde puedo pedir una beca para estudiar?,Las becas pueden solicitarse desde el portal d...,¿Dónde puedo ver mis notas?,¿Cómo solicito una beca?
7,¿Qué papeles tengo que presentar para inscribi...,Necesitas DNI y documentación académica previa...,¿Cómo me inscribo a materias?,¿Qué documentación necesito para inscribirme?
8,¿Cómo veo los horarios de mis materias?,El horario de cursada se encuentra disponible ...,¿Cómo puedo dar de baja una materia?,¿Dónde veo mi horario de cursada?
9,¿Cómo hago para abandonar una materia?,La baja de materias puede realizarse desde aut...,¿Cómo puedo dar de baja una materia?,¿Cómo puedo dar de baja una materia?


### Evaluación sobre el dataset completo

Se evaluaron los dos modelos de embeddings utilizando las 20 preguntas del dataset de evaluación.

Para considerar una recuperación correcta, se verificó que la pregunta recuperada por el modelo correspondiera semánticamente a la pregunta original del dataset de conocimiento.

Los resultados obtenidos fueron:

- all-MiniLM-L6-v2: 9 respuestas correctas sobre 20 consultas (45%).
- paraphrase-multilingual-MiniLM-L12-v2: 19 respuestas correctas sobre 20 consultas (95%).

Se observó que el modelo paraphrase-multilingual-MiniLM-L12-v2 logró recuperar correctamente consultas reformuladas mediante sinónimos y expresiones alternativas, mientras que el modelo all-MiniLM-L6-v2 presentó mayores dificultades para identificar equivalencias semánticas.

Por este motivo se selecciona paraphrase-multilingual-MiniLM-L12-v2 como modelo de embedding para la aplicación final, ya que ofrece una recuperación de información significativamente más precisa para consultas realizadas en lenguaje natural por estudiantes universitarios.

### Conclusiones

En este trabajo se desarrolló un chatbot basado en recuperación de información utilizando una base vectorial ChromaDB y un modelo LLM de Hugging Face. Como base de conocimiento se reutilizó el dataset construido en el TP4, incorporando en esta etapa una arquitectura de recuperación semántica mediante embeddings.

Se compararon dos modelos de embeddings: all-MiniLM-L6-v2 y paraphrase-multilingual-MiniLM-L12-v2. Para ello se construyó un dataset de evaluación independiente compuesto por preguntas reformuladas respecto del conjunto original, manteniendo el mismo significado semántico.

Los resultados obtenidos mostraron una diferencia significativa entre ambos modelos. Mientras que all-MiniLM-L6-v2 recuperó correctamente 9 de las 20 consultas evaluadas, paraphrase-multilingual-MiniLM-L12-v2 alcanzó 19 respuestas correctas sobre 20 consultas. Esto demuestra una mayor capacidad para reconocer equivalencias semánticas y consultas expresadas mediante sinónimos o formulaciones alternativas.

A partir de los resultados obtenidos, se seleccionó paraphrase-multilingual-MiniLM-L12-v2 como modelo de embeddings para la aplicación final, ya que ofrece una recuperación de información más precisa para consultas realizadas en lenguaje natural por estudiantes universitarios.

La incorporación de una base vectorial y un modelo LLM permitió evolucionar el chatbot desarrollado en el TP4 hacia una solución más flexible y robusta, capaz de comprender mejor las consultas de los usuarios y recuperar información relevante aun cuando las preguntas no coincidan literalmente con las almacenadas en la base de conocimiento.


### Referencias

-ChromaDB Documentation. https://docs.trychroma.com/

-Hugging Face Transformers Documentation. https://huggingface.co/docs/transformers

-Sentence Transformers Documentation. https://www.sbert.net/

-Modelo all-MiniLM-L6-v2. https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

-Modelo paraphrase-multilingual-MiniLM-L12-v2. https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

-Modelo Phi-3-mini-4k-instruct. https://huggingface.co/microsoft/Phi-3-mini-4k-instruct

-Material teórico y notebooks de la materia Procesamiento del Habla.

#### Uso de herramientas de IA generativa

Durante el desarrollo de este trabajo se utilizaron herramientas de inteligencia artificial generativa como apoyo para la comprensión de conceptos, la revisión de código, la resolución de dudas técnicas y la redacción de documentación.

Las soluciones propuestas fueron analizadas, adaptadas e integradas manualmente en el notebook, verificando su funcionamiento mediante pruebas sobre los datos utilizados en el trabajo práctico.

La utilización de estas herramientas tuvo como objetivo complementar el proceso de aprendizaje y facilitar la exploración de alternativas de implementación, manteniendo en todo momento la comprensión y validación de los resultados obtenidos.
